In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

# Load the dataset
df = pd.read_csv(r'c:\vscode\CVprojects\kenexaiHackathon\datasets\research_student.csv')

# Display basic info
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

Dataset Shape: (221, 24)

First few rows:
  Branch  Marks[10th]  Marks[12th]  Gender Board[10th] Board[12th] Category  \
0  CIVIL        77.57         64.6    Male  BSEB Patna  BSEB Patna      OBC   
1    CSE        86.40         71.8    Male        CBSE        CBSE      GEN   
2    CSE        88.14         78.0    Male        ICSE        ICSE      GEN   
3    CSE        65.40         59.8  Female        CBSE        CBSE       ST   
4    CSE        81.00         74.0    Male        CBSE        CBSE      GEN   

   GPA 1      Rank  Normalized Rank  ...  GPA 3  GPA 4  GPA 5  GPA 6  \
0   6.29   44718.0        15.970714  ...   5.94   5.41   6.25   6.13   
1   6.47   24222.0         8.650714  ...   5.88   5.53   6.44   6.19   
2   7.35   24723.0         8.829643  ...   6.54   6.41   6.50   6.69   
3   6.41  232157.0        82.913214  ...   5.71   5.24   5.88   6.25   
4   6.80   23252.0         8.304286  ...   5.88   6.00   5.93   5.44   

   Olympiads Qualified  Technical Projects  Tech Q

In [2]:
# Step 1: Clean column names
# Remove extra spaces, convert to lowercase, replace spaces with underscores
df.columns = (df.columns.str.strip()
              .str.lower()
              .str.replace(r'[\[\]]', '', regex=True)  # Remove brackets
              .str.replace(r'[^a-z0-9]+', '_', regex=True)  # Replace non-alphanumeric with underscore
              .str.strip('_'))

print("Cleaned column names:")
print(df.columns.tolist())
print("\nDataset shape after cleaning:", df.shape)

Cleaned column names:
['branch', 'marks10th', 'marks12th', 'gender', 'board10th', 'board12th', 'category', 'gpa_1', 'rank', 'normalized_rank', 'cgpa', 'current_back', 'ever_back', 'gpa_2', 'gpa_3', 'gpa_4', 'gpa_5', 'gpa_6', 'olympiads_qualified', 'technical_projects', 'tech_quiz', 'engg_coaching', 'ntse_scholarships', 'miscellany_tech_events']

Dataset shape after cleaning: (221, 24)


In [3]:
# Step 2: Check missing values and duplicates
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

print("\n\nDuplicate records:")
duplicates = df.duplicated().sum()
print(f"Total duplicates: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found")

# Show missing value percentage
print("\n\nMissing value percentage:")
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False) if missing_pct.sum() > 0 else "No missing values")

Missing values per column:
branch                    1
marks10th                 1
marks12th                 1
gender                    1
board10th                 1
board12th                 1
category                  1
gpa_1                     1
normalized_rank           1
cgpa                      1
current_back              1
ever_back                 2
gpa_2                     1
gpa_3                     1
gpa_4                     1
gpa_5                     1
gpa_6                     1
olympiads_qualified       1
technical_projects        1
tech_quiz                 1
engg_coaching             1
ntse_scholarships         1
miscellany_tech_events    1
dtype: int64


Duplicate records:
Total duplicates: 0
No duplicates found


Missing value percentage:
ever_back                 0.904977
gpa_2                     0.452489
ntse_scholarships         0.452489
engg_coaching             0.452489
tech_quiz                 0.452489
technical_projects        0.452489
olympiads_qualifi

In [4]:
# Step 3: Handle missing values
print("Handling missing values...\n")

# Fill numeric columns with median
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} with median: {median_val}")

# Fill categorical columns with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} with mode: {mode_val}")

print("\nMissing values after handling:")
print("Total missing values:", df.isnull().sum().sum())
print(f"\nDataset shape: {df.shape}")

Handling missing values...

Filled marks10th with median: 86.55
Filled marks12th with median: 79.1
Filled gpa_1 with median: 7.18
Filled normalized_rank with median: 10.694107142857142
Filled cgpa with median: 7.125
Filled current_back with median: 0.0
Filled ever_back with median: 0.0
Filled gpa_2 with median: 6.88
Filled gpa_3 with median: 6.82
Filled gpa_4 with median: 6.955
Filled gpa_5 with median: 7.63
Filled gpa_6 with median: 7.5
Filled olympiads_qualified with median: 4.0
Filled technical_projects with median: 2.0
Filled tech_quiz with median: 2.0
Filled engg_coaching with median: 3.0
Filled ntse_scholarships with median: 0.0
Filled miscellany_tech_events with median: 0.0
Filled branch with mode: ECE
Filled gender with mode: Male
Filled board10th with mode: CBSE
Filled board12th with mode: CBSE
Filled category with mode: GEN

Missing values after handling:
Total missing values: 0

Dataset shape: (221, 24)


C:\Users\Isha patel\AppData\Local\Temp\ipykernel_27872\1779374955.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
C:\Users\Isha patel\AppData\Local\Temp\ipykernel_27872\1779374955.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

In [5]:
# Step 4: Clean and standardize categorical columns
print("Cleaning categorical columns...\n")

# Clean categorical columns: strip whitespace, convert to lowercase
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    print(f"\n{col} unique values:")
    print(df[col].unique()[:10])  # Show first 10 unique values

# Convert to category dtype for memory efficiency
for col in categorical_cols:
    df[col] = df[col].astype('category')

print(f"\nCategorical columns converted to category dtype")
print(f"Dataset info after categorical cleaning:")
print(df.info())

Cleaning categorical columns...


branch unique values:
['civil' 'cse' 'ece' 'eee' 'it' 'mech' 'prod']

gender unique values:
['male' 'female']

board10th unique values:
['bseb patna' 'cbse' 'icse' 'bseb' 'c.b.s.e' 'b.s.e.b' 'i.c.s.e'
 'up board' 'isce' 'bihar board']

board12th unique values:
['bseb patna' 'cbse' 'icse' 'c.b.s.e' 'b.s.e.b' 'bseb' 'i.s.c' 'up board'
 'isc' 'i sc']

category unique values:
['obc' 'gen' 'st' 'sc']

Categorical columns converted to category dtype
Dataset info after categorical cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   branch                  221 non-null    category
 1   marks10th               221 non-null    float64 
 2   marks12th               221 non-null    float64 
 3   gender                  221 non-null    category
 4   board10th               221 non-null    cat

In [6]:
# Step 5: Feature Engineering
print("Creating new features...\n")

# Get numeric columns (excluding those with "marks" in name to avoid duplication)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Create academic average from 10th and 12th marks
if 'marks_10th' in df.columns and 'marks_12th' in df.columns:
    df['academic_average'] = (df['marks_10th'] + df['marks_12th']) / 2
    print("Created feature: academic_average = (marks_10th + marks_12th) / 2")

# Create GPA average from all GPA columns
gpa_cols = [col for col in df.columns if col.startswith('gpa') and col != 'gpa_1']
if gpa_cols:
    df['average_gpa'] = df[gpa_cols].mean(axis=1)
    print(f"Created feature: average_gpa from {gpa_cols}")

# Create engagement score from technical participation columns
engagement_cols = [col for col in df.columns if col in 
                  ['olympiads_qualified', 'technical_projects', 'tech_quiz', 
                   'engg_coaching', 'miscellany_tech_events']]
if engagement_cols:
    df['engagement_score'] = df[engagement_cols].sum(axis=1)
    print(f"Created feature: engagement_score = sum of {engagement_cols}")

# Create academic performance indicator
if 'cgpa' in df.columns and 'gpa_1' in df.columns:
    df['gpa_cgpa_ratio'] = df['gpa_1'] / (df['cgpa'] + 0.01)  # Avoid division by zero
    print("Created feature: gpa_cgpa_ratio")

print(f"\nNew features created. Dataset shape: {df.shape}")
print(f"New columns: {[col for col in df.columns if col in ['academic_average', 'average_gpa', 'engagement_score', 'gpa_cgpa_ratio']]}")

Creating new features...

Created feature: average_gpa from ['gpa_2', 'gpa_3', 'gpa_4', 'gpa_5', 'gpa_6']
Created feature: engagement_score = sum of ['olympiads_qualified', 'technical_projects', 'tech_quiz', 'engg_coaching', 'miscellany_tech_events']
Created feature: gpa_cgpa_ratio

New features created. Dataset shape: (221, 27)
New columns: ['average_gpa', 'engagement_score', 'gpa_cgpa_ratio']


In [7]:
# Step 6: Outlier Detection (IQR Method)
print("Detecting outliers using IQR method...\n")

outlier_cols = ['marks_10th', 'marks_12th', 'cgpa', 'normalized_rank', 'rank']
outlier_cols = [col for col in outlier_cols if col in df.columns]

outliers_count = 0
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers_in_col = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    if outliers_in_col > 0:
        print(f"{col}: {outliers_in_col} outliers (bounds: {lower_bound:.2f} - {upper_bound:.2f})")
        outliers_count += outliers_in_col
        
        # Cap outliers instead of removing them
        df[col] = df[col].clip(lower_bound, upper_bound)

print(f"\nTotal outliers detected: {outliers_count}")
print("Outliers capped to min/max bounds (not removed)")
print(f"Dataset shape after outlier handling: {df.shape}")

Detecting outliers using IQR method...

normalized_rank: 25 outliers (bounds: -0.64 - 23.87)
rank: 26 outliers (bounds: -2418.00 - 67894.00)

Total outliers detected: 51
Outliers capped to min/max bounds (not removed)
Dataset shape after outlier handling: (221, 27)


In [8]:
# Step 7: Normalize important numeric features
print("Normalizing numeric features using MinMaxScaler...\n")

# Get all numeric columns (from current DataFrame)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Filter to only numeric columns that should be normalized
normalize_cols = [col for col in numeric_cols if col not in 
                 ['rank', 'current_back', 'ever_back']]  # Exclude rank/count columns

if normalize_cols:
    scaler = MinMaxScaler()
    df[normalize_cols] = scaler.fit_transform(df[normalize_cols])
    print(f"Normalized {len(normalize_cols)} columns")
    print(f"Columns normalized: {normalize_cols}\n")
    
    print("Normalized data range (should be 0-1):")
    print(df[normalize_cols].describe())
else:
    print("No columns to normalize")

Normalizing numeric features using MinMaxScaler...

Normalized 19 columns
Columns normalized: ['marks10th', 'marks12th', 'gpa_1', 'normalized_rank', 'cgpa', 'gpa_2', 'gpa_3', 'gpa_4', 'gpa_5', 'gpa_6', 'olympiads_qualified', 'technical_projects', 'tech_quiz', 'engg_coaching', 'ntse_scholarships', 'miscellany_tech_events', 'average_gpa', 'engagement_score', 'gpa_cgpa_ratio']

Normalized data range (should be 0-1):
        marks10th   marks12th       gpa_1  normalized_rank        cgpa  \
count  221.000000  221.000000  221.000000       221.000000  221.000000   
mean     0.713697    0.528695    0.025715         0.416787    0.419593   
std      0.198169    0.227306    0.066608         0.269381    0.223914   
min      0.000000    0.000000    0.000000         0.000000    0.000000   
25%      0.589744    0.372796    0.013780         0.220536    0.240385   
50%      0.765734    0.561713    0.020598         0.329478    0.395833   
75%      0.869464    0.715365    0.029011         0.532322    0.5

In [9]:
# Step 8: Save cleaned dataset
print("Saving cleaned dataset...\n")

output_path = Path(r'c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets\research_student_cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"✓ Cleaned dataset saved: {output_path}")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumn summary:")
print(f"Categorical columns: {df.select_dtypes(include=['category']).columns.tolist()}")
print(f"Numeric columns: {len(df.select_dtypes(include=['float64', 'int64']).columns)}")

# Display final data info
print(f"\n\nFinal Dataset Info:")
print(df.info())

print(f"\n\nFirst few rows of cleaned data:")
print(df.head())

Saving cleaned dataset...

✓ Cleaned dataset saved: c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets\research_student_cleaned.csv

Final dataset shape: (221, 27)
Total columns: 27

Column summary:
Categorical columns: ['branch', 'gender', 'board10th', 'board12th', 'category']
Numeric columns: 22


Final Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   branch                  221 non-null    category
 1   marks10th               221 non-null    float64 
 2   marks12th               221 non-null    float64 
 3   gender                  221 non-null    category
 4   board10th               221 non-null    category
 5   board12th               221 non-null    category
 6   category                221 non-null    category
 7   gpa_1                   221 non-null    float64 
 8   rank                    2

In [10]:
# Step 9: Binary Encoding for Participation Columns
print("Converting participation columns to binary (0/1)...\n")

# These columns should be binary indicators (0 = No, 1 = Yes)
binary_cols = ['olympiads_qualified', 'technical_projects', 'tech_quiz', 
               'engg_coaching', 'ntse_scholarships', 'miscellany_tech_events']

for col in binary_cols:
    if col in df.columns:
        # Check current values
        unique_vals = df[col].unique()
        print(f"{col}: unique values {unique_vals}")
        
        # Convert to binary: any value > 0 becomes 1, 0 stays 0
        df[col] = (df[col] > 0).astype(int)

print("\nBinary columns after encoding:")
for col in binary_cols:
    if col in df.columns:
        print(f"{col}: {df[col].unique()}")

print(f"\nDataset shape: {df.shape}")

Converting participation columns to binary (0/1)...

olympiads_qualified: unique values [0.1 0.2 0.3 0.4 0.6 0.5 0.8 0.7 0.  0.9 1. ]
technical_projects: unique values [0.8 0.4 0.2 0.  0.6 1. ]
tech_quiz: unique values [0.42857143 0.         0.28571429 0.14285714 0.57142857 0.85714286
 0.71428571 1.        ]
engg_coaching: unique values [0.2 0.1 0.  0.3 0.4 0.6 0.5 0.7 0.8 1. ]
ntse_scholarships: unique values [0.         0.28571429 0.14285714 0.57142857 0.42857143 0.85714286
 1.         0.71428571]
miscellany_tech_events: unique values [1.  0.8 0.2 0.6 0.  0.4]

Binary columns after encoding:
olympiads_qualified: [1 0]
technical_projects: [1 0]
tech_quiz: [1 0]
engg_coaching: [1 0]
ntse_scholarships: [0 1]
miscellany_tech_events: [1 0]

Dataset shape: (221, 27)


In [11]:
# Step 10: Create Additional Features for ML
print("Creating additional features for ML...\n")

# Feature 1: Academic Performance Index
if 'cgpa' in df.columns and 'average_gpa' in df.columns:
    df['academic_performance_index'] = (df['cgpa'] + df['average_gpa']) / 2
    print("Created: academic_performance_index = (cgpa + average_gpa) / 2")
    print(f"  Min: {df['academic_performance_index'].min():.4f}")
    print(f"  Max: {df['academic_performance_index'].max():.4f}")
    print(f"  Mean: {df['academic_performance_index'].mean():.4f}\n")

# Feature 2: Total Backlogs
if 'current_back' in df.columns and 'ever_back' in df.columns:
    df['total_backlogs'] = df['current_back'] + df['ever_back']
    print("Created: total_backlogs = current_back + ever_back")
    print(f"  Min: {df['total_backlogs'].min():.0f}")
    print(f"  Max: {df['total_backlogs'].max():.0f}")
    print(f"  Mean: {df['total_backlogs'].mean():.2f}\n")

print(f"Dataset shape after feature creation: {df.shape}")
print(f"\nNew features added: academic_performance_index, total_backlogs")

Creating additional features for ML...

Created: academic_performance_index = (cgpa + average_gpa) / 2
  Min: 0.0027
  Max: 0.7356
  Mean: 0.2246

Created: total_backlogs = current_back + ever_back
  Min: 0
  Max: 14
  Mean: 1.51

Dataset shape after feature creation: (221, 29)

New features added: academic_performance_index, total_backlogs


In [12]:
# Step 11: Normalize new features and save final dataset
print("Normalizing new features and saving final dataset...\n")

# Normalize the new feature
if 'academic_performance_index' in df.columns:
    scaler_api = MinMaxScaler()
    df[['academic_performance_index']] = scaler_api.fit_transform(df[['academic_performance_index']])
    print("Normalized: academic_performance_index (range: 0-1)")

# Save updated cleaned dataset
output_path = Path(r'c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets\research_student_cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"\n✓ Final cleaned dataset saved: {output_path}")

# Display final summary
print(f"\nFinal Dataset Summary:")
print(f"  Shape: {df.shape}")
print(f"  Total columns: {len(df.columns)}")
print(f"  Missing values: {df.isnull().sum().sum()}")

# Show all columns
print(f"\nAll columns in final dataset:")
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    print(f"  {i:2d}. {col:40s} ({dtype})")

# Show new features summary
print(f"\n\nNew Features Added:")
print(f"  • academic_performance_index (normalized 0-1)")
print(f"  • total_backlogs (sum of current and ever backlogs)")
print(f"  • engagement_score (binary: 0/1)")
print(f"\nDataset ready for ML and merging!")

Normalizing new features and saving final dataset...

Normalized: academic_performance_index (range: 0-1)

✓ Final cleaned dataset saved: c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets\research_student_cleaned.csv

Final Dataset Summary:
  Shape: (221, 29)
  Total columns: 29
  Missing values: 0

All columns in final dataset:
   1. branch                                   (category)
   2. marks10th                                (float64)
   3. marks12th                                (float64)
   4. gender                                   (category)
   5. board10th                                (category)
   6. board12th                                (category)
   7. category                                 (category)
   8. gpa_1                                    (float64)
   9. rank                                     (float64)
  10. normalized_rank                          (float64)
  11. cgpa                                     (float64)
  12. current_back              